I plan to determine whether word frequency patterns can distinguish the writing styles ofdifferent authors. To do so, I will examine both Great Expectations and David Copperfield byCharles Dickens, and identify their similarities and differences in word frequency. Thesefindings will then be compared to The Adventures of Sherlock Holmes and The Hound of theBaskervilles by Arthur Conan Doyle, to see which words appear more often in the writing ofone author than the other.I chose two books per author on purpose. With only one book each, any difference I find couldjust mean "these two books are different" rather than "these two authors are different." Thesecond book by each author gives me a baseline for how much an author varies from himself, soI can check whether the gap between authors is actually larger than that. I also picked twoauthors who are harder to tell apart than usual: both are Victorian, both British, and allfour books are narrated in the first person, so era and narrative voice cannot explain anydifference I find.For this experiment, texts are collected from Project Gutenberg. Pre-processing includeslowercasing all tokens, removing punctuation and other unneeded symbols, and removing stopwords. Word frequencies are normalized per 10,000 words to make sure the comparison is fairregardless of the difference in the lengths of the books, since David Copperfield is about sixtimes as long as The Hound of the Baskervilles.

In [1]:
# Two books per author, so I can measure within-author variation as well as between-author.
# Downloaded from Project Gutenberg and saved next to this notebook:
#   Great Expectations           gutenberg.org/cache/epub/1400/pg1400.txt
#   David Copperfield            gutenberg.org/cache/epub/766/pg766.txt
#   Sherlock Holmes              gutenberg.org/cache/epub/1661/pg1661.txt
#   Hound of the Baskervilles    gutenberg.org/cache/epub/2852/pg2852.txt
files = {
    "Great Expectations":        "great_expectations.txt",
    "David Copperfield":         "david_copperfield.txt",
    "Sherlock Holmes":           "sherlock_holmes.txt",
    "Hound of the Baskervilles": "hound_of_the_baskervilles.txt",
}

author = {
    "Great Expectations": "Dickens",
    "David Copperfield": "Dickens",
    "Sherlock Holmes": "Doyle",
    "Hound of the Baskervilles": "Doyle",
}

# encoding="utf-8" matters here: these files use the typographic apostrophe, and reading
# them as plain ASCII would corrupt every contraction in the corpus.
raw_text = {}
for title, filename in files.items():
    with open(filename, encoding="utf-8") as f:
        raw_text[title] = f.read()

In [2]:
# The ebooks on Project Gutenberg start with a header and end with a licence footer that are
# not part of the original text. This function removes both.
# I had to strip these carefully: the footer is identical in all four files, so leaving it in
# would make every author look more similar than they really are.
def remove_gutenberg_header_footer(text):
    lines = text.splitlines()
    start_index = 0
    end_index = len(lines)

    for i, line in enumerate(lines):
        if "*** START OF THE PROJECT GUTENBERG EBOOK" in line.upper():
            # the start marker can wrap onto several lines, so skip to the one ending in ***
            j = i
            while j < len(lines):
                if lines[j].strip().endswith("***"):
                    start_index = j + 1
                    break
                j += 1
            break

    for i, line in enumerate(lines[start_index:], start=start_index):
        if "*** END OF THE PROJECT GUTENBERG EBOOK" in line.upper():
            end_index = i
            break

    return "\n".join(lines[start_index:end_index])


clean_text = {title: remove_gutenberg_header_footer(t) for title, t in raw_text.items()}

In [3]:
import re

# Convert text into word lists: lowercase everything and ignore punctuation.
def get_words(text):
    # These files use the typographic apostrophe (U+2019), not the ASCII one, so the regex
    # below split every contraction. "ain't" became "ain" + "t" and the name "Em'ly" became
    # "em" + "ly", and those fragments then showed up as some of Dickens's most distinctive
    # "words". Normalising the apostrophe first keeps contractions in one piece.
    text = text.replace("\u2019", "'")
    words = re.findall(r"[a-z']+", text.lower())
    # Anything left of length 1 is a leftover fragment, so drop it except for "a" and "i".
    return [w for w in words if len(w) > 1 or w in ["a", "i"]]


words = {title: get_words(t) for title, t in clean_text.items()}

for title in files:
    print(title, "-", len(words[title]), "words")

Great Expectations - 186601 words
David Copperfield - 358847 words
Sherlock Holmes - 105214 words
Hound of the Baskervilles - 59589 words


During this analysis, I want to examine the frequency both with and without the most commonEnglish words ("the", "and", "I", etc).

Word Frequency with Stop Words.

In [4]:
from collections import Counter

freq = {title: Counter(words[title]) for title in files}

for title in files:
    print(title + ":")
    print(freq[title].most_common(20))
    print()

Great Expectations:
[('the', 8145), ('and', 7096), ('i', 6490), ('to', 5156), ('of', 4438), ('a', 4049), ('in', 3028), ('that', 2988), ('was', 2836), ('it', 2671), ('he', 2209), ('you', 2189), ('had', 2093), ('my', 2069), ('me', 1998), ('his', 1860), ('as', 1775), ('with', 1760), ('at', 1637), ('on', 1420)]

David Copperfield:
[('the', 13703), ('i', 13204), ('and', 12309), ('to', 10456), ('of', 8696), ('a', 7954), ('in', 6240), ('was', 5315), ('that', 5233), ('my', 5215), ('it', 4751), ('her', 3874), ('me', 3620), ('he', 3526), ('you', 3506), ('with', 3369), ('as', 3197), ('had', 3059), ('said', 2950), ('his', 2935)]

Sherlock Holmes:
[('the', 5630), ('and', 3018), ('i', 3003), ('to', 2744), ('of', 2655), ('a', 2643), ('in', 1766), ('that', 1744), ('it', 1701), ('you', 1481), ('he', 1467), ('was', 1410), ('his', 1159), ('is', 1124), ('my', 1007), ('have', 924), ('as', 852), ('had', 831), ('with', 830), ('which', 771)]

Hound of the Baskervilles:
[('the', 3345), ('and', 1629), ('of', 16

Word Frequency Without Stop Words.

In [5]:
stop_words = {
    "the", "and", "of", "to", "a", "in", "that", "i", "it", "was",
    "he", "is", "his", "you", "as", "had", "with", "for", "on", "but",
    "not", "at", "by", "this", "be", "have", "from", "or", "we", "they",
    "an", "were", "which", "my", "me", "so", "if", "there", "would",
    "could", "should", "do", "did", "has", "been", "are", "him", "her",
    "she", "your", "all", "what", "no", "then", "out", "when", "into",
    "who", "our", "us", "will", "their", "them", "now",
    "some", "very", "up", "down", "one"
}

def remove_stop_words(word_list):
    return [w for w in word_list if w not in stop_words]


filtered = {title: remove_stop_words(words[title]) for title in files}
filtered_freq = {title: Counter(filtered[title]) for title in files}

for title in files:
    print(title + ":")
    print(filtered_freq[title].most_common(20))
    print()

Great Expectations:
[('said', 1349), ('mr', 711), ('joe', 691), ('more', 404), ('miss', 383), ('know', 382), ('come', 374), ('time', 373), ('little', 371), ('upon', 366), ('again', 359), ('like', 327), ('pip', 326), ('looked', 325), ('about', 320), ('never', 315), ('old', 312), ('much', 311), ('before', 309), ('man', 307)]

David Copperfield:
[('said', 2950), ('mr', 2488), ('little', 1096), ('am', 866), ('upon', 806), ('know', 794), ('more', 787), ('micawber', 771), ('aunt', 763), ('miss', 715), ('peggotty', 710), ('mrs', 673), ('time', 670), ('any', 669), ('never', 662), ('about', 660), ('much', 657), ('old', 642), ('like', 641), ('made', 627)]

Sherlock Holmes:
[('said', 486), ('upon', 465), ('holmes', 445), ('man', 291), ('mr', 275), ('little', 269), ('see', 229), ('well', 201), ('may', 197), ('am', 185), ('over', 183), ('more', 174), ('think', 174), ('room', 171), ('know', 170), ('shall', 169), ('can', 168), ('about', 168), ('before', 164), ('must', 161)]

Hound of the Baskervilles

The two Dickens books share 13 of their top 20 words: said, mr, more, miss, know, time, little,upon, about, never, old, much and like. "Said" is the single most frequent word in both. Thetwo Doyle books share 12 of theirs: upon, said, holmes, man, more, know, over, see, well,about, may and can. Once the character names are ignored, an author's frequent vocabularylooks stable across his own books.Comparing across authors is less obvious than I expected, because both writers use most of thesame words, and said, upon, know, more and about appear on all four lists. The difference showsup in how heavily each word is used, so the next step is to normalize and compare ratesdirectly.

In [6]:
# Rates per 10,000 words, so the different book lengths do not distort the comparison.
def freq_per_10000(title, word):
    return filtered_freq[title][word] / len(filtered[title]) * 10000


shared_words = ["said", "upon", "man", "little", "know", "see"]

for word in shared_words:
    print(word)
    for title in files:
        print("  " + title + " (" + author[title] + "):", round(freq_per_10000(title, word), 2))
    print()

said
  Great Expectations (Dickens): 148.39
  David Copperfield (Dickens): 165.61
  Sherlock Holmes (Doyle): 96.21
  Hound of the Baskervilles (Doyle): 83.5

upon
  Great Expectations (Dickens): 40.26
  David Copperfield (Dickens): 45.25
  Sherlock Holmes (Doyle): 92.05
  Hound of the Baskervilles (Doyle): 109.6

man
  Great Expectations (Dickens): 33.77
  David Copperfield (Dickens): 21.05
  Sherlock Holmes (Doyle): 57.61
  Hound of the Baskervilles (Doyle): 69.93

little
  Great Expectations (Dickens): 40.81
  David Copperfield (Dickens): 61.53
  Sherlock Holmes (Doyle): 53.25
  Hound of the Baskervilles (Doyle): 19.83

know
  Great Expectations (Dickens): 42.02
  David Copperfield (Dickens): 44.58
  Sherlock Holmes (Doyle): 33.65
  Hound of the Baskervilles (Doyle): 41.05

see
  Great Expectations (Dickens): 30.14
  David Copperfield (Dickens): 31.33
  Sherlock Holmes (Doyle): 45.33
  Hound of the Baskervilles (Doyle): 39.32



This is where the two authors separate. "Upon" runs at 40.3 and 45.3 per 10,000 in the twoDickens books but 92.1 and 109.6 in the two Doyle books, so each author's own two values sitclose together while the gap between authors is large. "Said" does the same thing in reverse,148.4 and 165.6 for Dickens against 96.2 and 83.5 for Doyle."Little" is a warning. Its four values are 40.8, 61.5, 53.3 and 19.8, which do not group byauthor at all: The Hound is the lowest and David Copperfield the highest even though those twohave different authors. A word can differ a lot between books and still be useless for tellingwriters apart, which is the reason I kept a second book per author.

Next I want to find the words that most separate the two authors, rather than checking words Ipicked by hand. I pool each author's two books, compute the rate per 10,000 for every word,and rank by the ratio between the two authors. Character and place names are excluded, sincenames like "Pip" or "Holmes" identify the book rather than the writer.

In [7]:
import math

names = set('''
pip joe estella havisham biddy herbert wemmick jaggers magwitch pumblechook orlick
compeyson provis gargery trabb startop drummle camilla georgiana clara wopsle
david davy copperfield murdstone peggotty micawber betsey trotwood steerforth uriah heep
agnes wickfield dora spenlow traddles barkis emly emily em'ly creakle littimer gummidge
chillip dick mell ham
jorkins crupp markleham annie sophy janet omer jip mills trot dartle rosa
holmes watson sherlock lestrade adler irene moriarty mycroft hudson baskerville henry
stapleton mortimer barrymore selden frankland lyons grimpen devonshire dartmoor
simon rucastle mccarthy openshaw hosmer windibank roylott stoner coram charles
london england french paris norfolk yarmouth canterbury
'''.split())

pooled = {}
pooled_total = {}
for a in ["Dickens", "Doyle"]:
    books = [t for t in files if author[t] == a]
    pooled[a] = Counter()
    for t in books:
        pooled[a] += filtered_freq[t]
    pooled_total[a] = sum(len(filtered[t]) for t in books)

scores = []
for word in set(pooled["Dickens"]) | set(pooled["Doyle"]):
    d, c = pooled["Dickens"][word], pooled["Doyle"][word]
    # also catch plurals and possessives ("micawber's" -> "micawber")
    if d + c < 25 or word in names or word.rstrip("s") in names or word.split("'")[0] in names:
        continue
    rd = d / pooled_total["Dickens"] * 10000
    rc = c / pooled_total["Doyle"] * 10000
    # +0.5 smoothing so a word missing from one author gives a finite ratio
    scores.append((math.log2((rd + 0.5) / (rc + 0.5)), word, rd, rc))

scores.sort(reverse=True)

print("Used much more by Dickens:")
for s, word, rd, rc in scores[:12]:
    print("  %-12s Dickens %6.2f   Doyle %6.2f   %5.1fx" % (word, rd, rc, 2 ** s))

print("\nUsed much more by Doyle:")
for s, word, rd, rc in scores[-12:][::-1]:
    print("  %-12s Dickens %6.2f   Doyle %6.2f   %5.1fx" % (word, rd, rc, 2 ** -s))

Used much more by Dickens:
  aunt         Dickens  28.73   Doyle   0.25    38.9x
  replied      Dickens  10.18   Doyle   0.38    12.2x
  ain't        Dickens   5.28   Doyle   0.00    11.6x
  coach        Dickens   5.54   Doyle   0.13     9.6x
  mas'r        Dickens   4.31   Doyle   0.00     9.6x
  ma'am        Dickens   4.09   Doyle   0.00     9.2x
  parlour      Dickens   3.83   Doyle   0.00     8.7x
  she's        Dickens   3.09   Doyle   0.00     7.2x
  couldn't     Dickens   5.39   Doyle   0.38     6.7x
  anybody      Dickens   3.53   Doyle   0.13     6.4x
  doen't       Dickens   2.68   Doyle   0.00     6.4x
  umble        Dickens   2.53   Doyle   0.00     6.1x

Used much more by Doyle:
  moor         Dickens   0.04   Doyle  20.57    39.2x
  dr           Dickens   0.00   Doyle  17.54    36.1x
  hound        Dickens   0.19   Doyle   9.34    14.3x
  cab          Dickens   0.04   Doyle   5.93    12.0x
  baronet      Dickens   0.04   Doyle   5.43    11.0x
  crime        Dickens   0.04

So yes, there are words strongly preferred by one author. Dickens uses domestic words such asaunt, replied, coach and parlour, along with contractions and dialect spellings from hisworking class characters, such as ain't, ma'am, mas'r, doen't and umble. Doyle uses thevocabulary of detective stories: moor, hound, cab, crime, police, baronet, facts andinspector. Several of these appear zero times in the other author, and the ratios reach about39 times.One thing I should be honest about is that a lot of this is subject matter rather than writingstyle. Detective stories contain crimes and Dickens novels contain aunts, so some of this gapwould disappear if Dickens had written a mystery. The words that look most like genuine styleare "replied" and the dialect spellings.

### Extra creditEach word in a book can be treated as one trial that either is or is not the word I amchecking, so the count follows a binomial distribution, X ~ Binomial(N, p), where p isestimated as N(word) / N(words). To test whether two books really differ I use a twoproportion z-test on the pooled rate.I run each test twice, once within an author and once between authors. The within-author testis the control: a word is only a useful marker if it stays steady inside one author and shiftsbetween authors.

In [8]:
import statistics
from scipy import stats

def z_test(word, title1, title2):
    x1, n1 = filtered_freq[title1][word], len(filtered[title1])
    x2, n2 = filtered_freq[title2][word], len(filtered[title2])
    p1, p2 = x1 / n1, x2 / n2
    pooled_p = (x1 + x2) / (n1 + n2)
    z = (p1 - p2) / math.sqrt(pooled_p * (1 - pooled_p) * (1 / n1 + 1 / n2))
    return z, 2 * stats.norm.sf(abs(z))


pairs = [
    ("within Dickens", "Great Expectations", "David Copperfield"),
    ("within Doyle",   "Sherlock Holmes",    "Hound of the Baskervilles"),
    ("across authors", "Great Expectations", "Sherlock Holmes"),
    ("across authors", "David Copperfield",  "Hound of the Baskervilles"),
]

within_z, across_z = [], []
for word in ["upon", "said", "little", "about"]:
    print(word)
    for kind, t1, t2 in pairs:
        z, p = z_test(word, t1, t2)
        flag = "significant" if p < 0.05 else "not significant"
        print("  %-15s z = %7.2f   p = %8.5f   %s" % (kind, z, p, flag))
        (within_z if kind.startswith("within") else across_z).append(abs(z))
    print()

print("median |z| within an author :", round(statistics.median(within_z), 2))
print("median |z| across authors   :", round(statistics.median(across_z), 2))

upon
  within Dickens  z =   -1.86   p =  0.06306   not significant
  within Doyle    z =   -2.41   p =  0.01614   significant
  across authors  z =  -12.21   p =  0.00000   significant
  across authors  z =  -13.79   p =  0.00000   significant

said
  within Dickens  z =   -3.37   p =  0.00075   significant
  within Doyle    z =    1.81   p =  0.07106   not significant
  across authors  z =    8.31   p =  0.00000   significant
  across authors  z =   10.48   p =  0.00000   significant

little
  within Dickens  z =   -6.90   p =  0.00000   significant
  within Doyle    z =    7.07   p =  0.00000   significant
  across authors  z =   -3.34   p =  0.00084   significant
  across authors  z =    8.81   p =  0.00000   significant

about
  within Dickens  z =   -0.75   p =  0.45044   not significant
  within Doyle    z =   -0.76   p =  0.44929   not significant
  across authors  z =    0.60   p =  0.55048   not significant
  across authors  z =    0.14   p =  0.89260   not significant

media

The binomial distributions do differ much more between authors than within one. For "upon" thewithin-author tests give z = -1.86 and z = -2.41, which is barely any movement, while thebetween-author tests give z = -12.21 and z = -13.79. "Said" behaves the same way, although italso differs within Dickens (z = -3.37), so it is not quite as clean.The two failures are the more interesting result. "Little" is significant inside both authors,z = -6.90 for Dickens and z = 7.07 for Doyle, which are larger than one of its between-authorvalues (z = -3.34), so it is measuring the individual book rather than the writer. "About" is the oppositecase and does not move anywhere, within or between. Because each book gives tens of thousandsof trials, almost any real difference becomes statistically significant, so a small p-value onits own does not show anything about authorship. What matters is whether the between-authordifference is larger than the within-author one, which is what the two medians at the endcompare.### ConclusionWord frequency can distinguish these two authors, but not in the way I first assumed. Eachauthor is consistent with himself: the two Dickens books share 13 of their top 20 words atsimilar rates, and the two Doyle books share 12. Across authors the same common words appear,and what changes is how heavily each is used: "upon" runs at 43.6 per 10,000 for Dickensagainst 98.4 for Doyle, and "said" at 159.8 against 91.6. The most extreme words by ratio are largelyexplained by genre rather than style, and the statistical tests only support authorship oncethey are read against the within-author control instead of by p-value alone.